# Placement transform preview: A vs. synthesized target

Takes a real RealWorld `forearm` recording -- treated as if it had been recorded
at SMPL placement A (`experiments/scripts/placements/forearm_placements.yaml`)
-- and applies the SMPL-derived rotation to synthesize what the same physical
motion would have looked like recorded at a chosen target placement. Plots
accelerometer and gyroscope, all three axes, both placements, so the effect of
the rotation on the raw signal is directly visible.

The target is picked in the cell below -- either an index into
`forearm_placements.yaml` (1=B, 2=C, 3=D, ...) or a path to a standalone
single-sensor YAML, exactly as saved by `place_sensors.py`, so a brand-new
placement can be previewed immediately without merging it into
`forearm_placements.yaml` first.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

EXPERIMENTS_DIR = Path.cwd()  # this notebook lives directly in scripts/
sys.path.insert(0, str(EXPERIMENTS_DIR))
sys.path.insert(0, str(EXPERIMENTS_DIR.parent / "preprocessing"))
sys.path.insert(0, str(EXPERIMENTS_DIR.parent / "models"))

from loading import load_sensor_parts
import config
from synthesize_placements import (
    load_rotation_A_to_target, target_tag, rotation_A_to_file, rotate_windows,
)

# --- Choose the target placement here ---
# int   -> index into forearm_placements.yaml's sensors list (0=A is the
#          reference itself, 1=B, 2=C, 3=D, ...)
# str   -> path to a standalone single-sensor YAML, freshly saved by
#          place_sensors.py before it's been merged into forearm_placements.yaml
TARGET = 3  # placement D

if isinstance(TARGET, int):
    R_A_to_target = load_rotation_A_to_target(TARGET)
    target_label = target_tag(TARGET)
else:
    R_A_to_target, target_angle_deg, target_label = rotation_A_to_file(TARGET)

print(f"Target: {target_label}")
print("Rotation A -> target:")
print(np.round(R_A_to_target, 5))

## Load a real recording (treated as placement A) and resample acc+gyr onto a common grid

`subject_1 / running / forearm` is used as the example -- running has clear
periodic structure, which makes the effect of a rotation on the raw signal easy
to see. Only the first 8 seconds are used, resampled to a uniform 50 Hz grid
(matching the windowing pipeline's resampling approach) so the two channels are
sample-aligned before we rotate them.

In [ ]:
from windowing import _resample_xyz

subject, activity, position = "subject_1", "running", "forearm"
duration_s = 8.0
target_hz = config.TARGET_HZ
n_samples = int(duration_s * target_hz)

acc_part = load_sensor_parts(config.DATA_ROOT / subject, activity, "acc", position)[0]
gyr_part = load_sensor_parts(config.DATA_ROOT / subject, activity, "gyr", position)[0]

t0 = max(acc_part["time_ms"].iloc[0], gyr_part["time_ms"].iloc[0]) / 1000.0
query = np.linspace(t0, t0 + duration_s, n_samples, endpoint=False)

acc_A = _resample_xyz(acc_part, query)  # (n_samples, 3)
gyr_A = _resample_xyz(gyr_part, query)  # (n_samples, 3)
X_A = np.concatenate([acc_A, gyr_A], axis=1)  # (n_samples, 6): [acc_x,y,z, gyr_x,y,z]

print(f"Loaded {subject}/{activity}/{position}, first {duration_s}s -> X_A shape {X_A.shape}")

## Apply the rotation: synthesize the target placement from real placement A

`rotate_windows` applies the same 3x3 rotation `R_A_to_target` independently to
the acc block and the gyr block -- both modalities transform identically.

In [ ]:
X_target = rotate_windows(X_A, R_A_to_target)
acc_target, gyr_target = X_target[:, 0:3], X_target[:, 3:6]

t_axis = query - query[0]  # seconds since start, for plotting

## Plot: placement A (real) vs. target placement (synthesized), all 3 axes, both sensors

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

AXIS_COLORS = {"x": "#e02020", "y": "#20c050", "z": "#2060e0"}

fig = make_subplots(
    rows=2, cols=2,
    shared_xaxes=True,
    subplot_titles=("Accelerometer", "Gyroscope", "", ""),
    vertical_spacing=0.06, horizontal_spacing=0.06,
)

rows_data = [("A (real)", acc_A, gyr_A), (f"{target_label} (synthesized)", acc_target, gyr_target)]
axis_labels = ("x", "y", "z")

for row_idx, (label, acc, gyr) in enumerate(rows_data):
    r = row_idx + 1
    for i, axis_label in enumerate(axis_labels):
        show_legend = (row_idx == 0)
        fig.add_trace(go.Scatter(
            x=t_axis, y=acc[:, i], mode="lines", name=axis_label,
            line=dict(color=AXIS_COLORS[axis_label], width=1.1),
            legendgroup=axis_label, showlegend=show_legend,
        ), row=r, col=1)
        fig.add_trace(go.Scatter(
            x=t_axis, y=gyr[:, i], mode="lines", name=axis_label,
            line=dict(color=AXIS_COLORS[axis_label], width=1.1),
            legendgroup=axis_label, showlegend=False,
        ), row=r, col=2)

fig.update_xaxes(title_text="seconds", row=2, col=1)
fig.update_xaxes(title_text="seconds", row=2, col=2)

fig.update_layout(
    template="plotly_white",
    width=760, height=370,
    margin=dict(l=55, r=10, t=42, b=32),
    font=dict(family="Arial, Helvetica, sans-serif", size=10),
    legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="left", x=0, font=dict(size=8)),
)
fig.show()

plots_dir = EXPERIMENTS_DIR / "plots"
plots_dir.mkdir(exist_ok=True)

pdf_path = plots_dir / f"placement_transform_preview_{target_label}.pdf"
fig.write_image(pdf_path)
print(f"Saved -> {pdf_path}")

png_path = plots_dir / f"placement_transform_preview_{target_label}.png"
fig.write_image(png_path, scale=3)
print(f"Saved -> {png_path}")

## Sanity check: rotation preserves magnitude

A rotation only changes how a vector is *expressed*, never its length. `|acc_A|`
and `|acc_target|` (and the gyro equivalent) should be identical at every
timestep -- this is what "the same physical measurement, just in a different
basis" looks like numerically, not just conceptually.

In [ ]:
acc_mag_A, acc_mag_target = np.linalg.norm(acc_A, axis=1), np.linalg.norm(acc_target, axis=1)
gyr_mag_A, gyr_mag_target = np.linalg.norm(gyr_A, axis=1), np.linalg.norm(gyr_target, axis=1)

print("max |acc_A - acc_target| magnitude difference:", np.abs(acc_mag_A - acc_mag_target).max())
print("max |gyr_A - gyr_target| magnitude difference:", np.abs(gyr_mag_A - gyr_mag_target).max())

fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))
axes[0].plot(t_axis, acc_mag_A, label="|acc_A|", linewidth=1.2)
axes[0].plot(t_axis, acc_mag_target, label=f"|acc_{target_label}|", linewidth=1.2, linestyle="--")
axes[0].set_title("Accelerometer magnitude")
axes[0].set_ylabel("m/s^2")
axes[0].legend()

axes[1].plot(t_axis, gyr_mag_A, label="|gyr_A|", linewidth=1.2)
axes[1].plot(t_axis, gyr_mag_target, label=f"|gyr_{target_label}|", linewidth=1.2, linestyle="--")
axes[1].set_title("Gyroscope magnitude")
axes[1].set_ylabel("rad/s")
axes[1].legend()

for ax in axes:
    ax.set_xlabel("seconds")
plt.tight_layout()
plt.show()